# Local DPO fine-tuning and Ollama deployment

This notebook trains a **LoRA adapter locally** with Hugging Face TRL's DPOTrainer, merges it into the downloaded base model, then imports the merged Safetensors model into Ollama. Ollama runs the result locally; it does not perform DPO training itself.

The provided `pharma_preference_data.csv` is used by default. It currently has only a few examples, so treat the result as a workflow demo?not a useful medical model. Collect and review substantially more high-quality preference pairs before relying on training results.

## Before you start (Windows)

1. Install and start [Ollama](https://ollama.com/download).
2. Use Python 3.10+ in a virtual environment. Install a PyTorch build appropriate for your CPU/GPU from [pytorch.org](https://pytorch.org/get-started/locally/), then run `%pip install -U transformers datasets peft trl accelerate bitsandbytes` in the notebook.
3. An NVIDIA CUDA GPU is strongly recommended. The default 4-bit QLoRA path needs CUDA and bitsandbytes; set `USE_4BIT = False` below to use standard LoRA instead. CPU training is supported but very slow and memory-intensive.
4. Hugging Face downloads the base model automatically. No Hugging Face token is required for the selected model.


In [1]:
%pip install -U transformers datasets peft trl accelerate bitsandbytes sentencepiece


  Using cached numpy-2.5.3-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.6-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 3.0 MB/s eta 0:00:04
   ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [6]:
%pip install sentencepiece


Note: you may need to restart the kernel to use updated packages.


In [21]:
from pathlib import Path
import gc
import subprocess
import torch
from datasets import load_dataset
from huggingface_hub.constants import HF_HUB_CACHE
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OLLAMA_MODEL_NAME = "pharma-qwen-dpo"
OUTPUT_DIR = Path("runs") / "qwen2.5-1.5b-dpo"
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_MODEL_DIR = OUTPUT_DIR / "ollama-model"
DATA_SOURCE = "local"  # Change to "huggingface" to use a Hub preference dataset.
HF_DATASET_ID = "trl-lib/ultrafeedback_binarized"
HF_DATASET_SPLIT = "train"
MAX_TRAIN_SAMPLES = None  # For a Hub dataset, optionally set e.g. 1000.
USE_4BIT = torch.cuda.is_available()

model_cache_dir = Path(HF_HUB_CACHE) / f"models--{BASE_MODEL.replace('/', '--')}"
print(f"Base model: {BASE_MODEL}")
print(f"Base model cache folder: {model_cache_dir}")
print(f"Training output folder: {OUTPUT_DIR.resolve()}")
print(f"CUDA available: {torch.cuda.is_available()}; 4-bit QLoRA: {USE_4BIT}")


Base model: Qwen/Qwen2.5-1.5B-Instruct
Base model cache folder: C:\Users\sridh\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct
Training output folder: C:\Users\sridh\OneDrive\Documents\LLM learning\Complete-LLM-Finetuning\LLM Fine-Tuning-16-Preference-based-training\runs\qwen2.5-1.5b-dpo
CUDA available: False; 4-bit QLoRA: False


## Load and validate preference pairs

DPO expects `prompt`, `chosen`, and `rejected` fields. The local CSV already has these columns. You can switch `DATA_SOURCE` above to `"huggingface"` to download the optional Hub dataset instead.


In [3]:
if DATA_SOURCE == "local":
    candidates = [
        Path("pharma_preference_data.csv"),
        Path("LLM Fine-Tuning-16-Preference-based-training") / "pharma_preference_data.csv",
    ]
    dataset_path = next((candidate for candidate in candidates if candidate.is_file()), None)
    if dataset_path is None:
        raise FileNotFoundError(
            "Could not find pharma_preference_data.csv. Run the notebook from the repository root "
            "or the LLM Fine-Tuning-16-Preference-based-training folder."
        )
    train_dataset = load_dataset("csv", data_files=str(dataset_path), split="train")
elif DATA_SOURCE == "huggingface":
    train_dataset = load_dataset(HF_DATASET_ID, split=HF_DATASET_SPLIT)
else:
    raise ValueError("DATA_SOURCE must be 'local' or 'huggingface'.")

required_columns = {"prompt", "chosen", "rejected"}
missing_columns = required_columns.difference(train_dataset.column_names)
if missing_columns:
    raise ValueError(f"Dataset is missing required DPO columns: {sorted(missing_columns)}")

def is_valid_preference_field(value):
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, list) and value:
        return all(
            isinstance(message, dict)
            and isinstance(message.get("role"), str)
            and isinstance(message.get("content"), str)
            and bool(message["content"].strip())
            for message in value
        )
    return False

for index, example in enumerate(train_dataset):
    if any(not is_valid_preference_field(example[field]) for field in required_columns):
        raise ValueError(f"Dataset row {index} has an invalid prompt/chosen/rejected value.")

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))
if len(train_dataset) < 100:
    print(f"Warning: only {len(train_dataset)} preference pairs are loaded; this is a small demo dataset.")

print(f"Loaded {len(train_dataset)} preference pairs")
print(train_dataset[0])


Generating train split: 5 examples [00:00, 746.03 examples/s]

Loaded 5 preference pairs
{'prompt': 'Explain the mechanism of action of Metformin in simple scientific terms.', 'chosen': 'Metformin activates AMPK, which helps cells use glucose more efficiently, reduces glucose production by the liver, and improves insulin sensitivity without increasing insulin levels.', 'rejected': 'Metformin only lowers blood sugar by stopping sugar absorption from food and does not affect other organs.'}


## Download the base model and attach a LoRA adapter

The Hugging Face model is downloaded to the local Transformers cache on first use. With CUDA, the notebook uses 4-bit NF4 QLoRA by default to reduce VRAM use. Without CUDA, it loads the model in FP32 and trains standard LoRA on CPU.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if USE_4BIT and not torch.cuda.is_available():
    raise RuntimeError("USE_4BIT=True requires a CUDA GPU. Set USE_4BIT=False for standard LoRA.")

compute_dtype = (
    torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16 if torch.cuda.is_available()
    else torch.float32
)
model_kwargs = {"torch_dtype": compute_dtype}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["device_map"] = "cpu"

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = quantization_config

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.config.use_cache = False
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)
else:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sridh\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\sridh\anaconda3\envs\l

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


## Run DPO training

The starter settings keep the run small. Increase training data and adjust batch size/sequence length to match available memory. Training outputs and adapter checkpoints are saved under `runs/qwen2.5-1.5b-dpo`.


In [18]:
import inspect

config_values = {
    "output_dir": str(OUTPUT_DIR),
    "beta": 0.1,
    "learning_rate": 5e-6,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 1,
    "max_length": 512,
    "max_prompt_length": 256,
    "logging_steps": 1,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "report_to": "none",
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "fp16": torch.cuda.is_available() and compute_dtype == torch.float16,
    "bf16": torch.cuda.is_available() and compute_dtype == torch.bfloat16,
}

supported_args = inspect.signature(DPOConfig).parameters
unsupported_args = sorted(set(config_values) - set(supported_args))
if unsupported_args:
    print(
        "Installed TRL/Transformers does not support these DPOConfig settings; "
        f"using library defaults for them: {unsupported_args}"
    )

training_args = DPOConfig(
    **{name: value for name, value in config_values.items() if name in supported_args}
)
print(f"DPOConfig ready (TRL {__import__('trl').__version__}).")


Installed TRL/Transformers does not support these DPOConfig settings; using library defaults for them: ['max_prompt_length']
DPOConfig ready (TRL 1.14.1).


In [10]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
train_result = trainer.train()
print(train_result.metrics)

# Persist the adapter immediately so merging can resume after a kernel restart.
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved trained adapter and tokenizer to: {ADAPTER_DIR.resolve()}")


Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer i

Step,Training Loss
1,0.693147
2,0.692353


{'train_runtime': 63.1794, 'train_samples_per_second': 0.079, 'train_steps_per_second': 0.032, 'total_flos': 4273438089216.0, 'train_loss': 0.6927502453327179, 'epoch': 1.0}


## Save and merge the trained adapter

### What gets downloaded and where

1. `BASE_MODEL` is `Qwen/Qwen2.5-1.5B-Instruct`. `from_pretrained()` downloads its weights and tokenizer from Hugging Face on the first run. The default Windows cache is under `%USERPROFILE%\\.cache\\huggingface\\hub\\models--Qwen--Qwen2.5-1.5B-Instruct` (unless `HF_HOME` or `HF_HUB_CACHE` is set). The download is cached; it is not copied into this project folder.
2. DPO training updates the LoRA adapter, not the full base model. The adapter and tokenizer are saved to `runs/qwen2.5-1.5b-dpo/adapter`.
3. For merging, this notebook reloads the original base weights from the Hugging Face cache, then `PeftModel.from_pretrained()` attaches the trained adapter.
4. `merge_and_unload()` folds the adapter weight deltas into the base weights. Conceptually, each adapted weight is `W_merged = W_base + delta_W`; the result is a standalone model rather than a separate adapter.
5. The merged Safetensors model is saved under a new timestamped folder inside `runs/qwen2.5-1.5b-dpo/`. A new folder per run avoids overwriting shards that Windows or another process may have open.
6. The next Ollama cell imports that exact merged folder using its generated `Modelfile`. The Hugging Face cache is the source; the timestamped `ollama-model-*` folder is the export Ollama reads.

Run the cells in order: load data, load model and attach LoRA, create DPO config, train, save and merge, then create and test the Ollama model. Training and merging can require substantial disk space and RAM.



In [19]:
from datetime import datetime

ADAPTER_CONFIG = ADAPTER_DIR / "adapter_config.json"
if "trainer" in globals():
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
elif not ADAPTER_CONFIG.is_file():
    raise RuntimeError(
        "No trained adapter is available. Run the DPO training cell first; it saves the adapter "
        f"to {ADAPTER_DIR.resolve()}. If the kernel was restarted, rerun training to create it."
    )
elif "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)

print(f"Using trained adapter from: {ADAPTER_DIR.resolve()}")

# Release training memory when the trainer is still present in this kernel.
if "trainer" in globals():
    del trainer
if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

merge_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
base_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=merge_dtype,
    device_map="cpu",
    low_cpu_mem_usage=True,
)
adapter_model = PeftModel.from_pretrained(base_for_merge, ADAPTER_DIR)
merged_model = adapter_model.merge_and_unload()

# Use a fresh directory on each run rather than replacing possibly locked Windows files.
run_stamp = datetime.now().strftime("%Y%m%d-%H%M%S-%f")
MERGED_MODEL_DIR = OUTPUT_DIR / f"ollama-model-{run_stamp}"
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=False)
merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_MODEL_DIR)
print(f"Merged model saved for Ollama import at: {MERGED_MODEL_DIR.resolve()}")
del adapter_model, merged_model, base_for_merge
gc.collect()


Using trained adapter from: C:\Users\sridh\OneDrive\Documents\LLM learning\Complete-LLM-Finetuning\LLM Fine-Tuning-16-Preference-based-training\runs\qwen2.5-1.5b-dpo\adapter


Writing model shards: 100%|██████████| 4/4 [01:05<00:00, 16.49s/it]


Merged model saved for Ollama import at: C:\Users\sridh\OneDrive\Documents\LLM learning\Complete-LLM-Finetuning\LLM Fine-Tuning-16-Preference-based-training\runs\qwen2.5-1.5b-dpo\ollama-model-20261005-202227-444631


676

## Convert the merged model to GGUF, then import into Ollama

Ollama's Safetensors importer in this setup rejects the `Qwen2ForCausalLM` architecture. Convert the merged Hugging Face model to GGUF with `llama.cpp`, which supports Qwen2.5, and import the GGUF file instead.

On Windows PowerShell, clone `llama.cpp` and install its Python conversion requirements once:

```powershell
git clone https://github.com/ggml-org/llama.cpp "$env:USERPROFILE\llama.cpp"
python -m pip install -r "$env:USERPROFILE\llama.cpp\requirements.txt"
```

Then run the code cell below. It creates a full-precision F16 GGUF file under `runs/qwen2.5-1.5b-dpo` and imports that file into Ollama. The F16 export uses several gigabytes of disk space; quantization can be added later to reduce runtime memory.


In [7]:
from pathlib import Path
import subprocess
import sys


def run_captured_utf8(args, **kwargs):
    return subprocess.run(
        args,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        **kwargs,
    )


OLLAMA_MODEL_NAME = globals().get("OLLAMA_MODEL_NAME", "pharma-qwen-dpo")
output_dir_relative = Path("runs") / "qwen2.5-1.5b-dpo"
output_dir_candidates = [
    output_dir_relative,
    Path("LLM Fine-Tuning-16-Preference-based-training") / output_dir_relative,
]
OUTPUT_DIR = Path(
    globals().get(
        "OUTPUT_DIR",
        next((candidate for candidate in output_dir_candidates if candidate.is_dir()), output_dir_relative),
    )
)
MERGED_MODEL_DIR = Path(globals().get("MERGED_MODEL_DIR", OUTPUT_DIR / "ollama-model"))


def is_complete_merged_model(path):
    return (path / "config.json").is_file() and any(path.glob("*.safetensors"))

if not is_complete_merged_model(MERGED_MODEL_DIR):
    completed_exports = [
        path for path in OUTPUT_DIR.glob("ollama-model-*")
        if is_complete_merged_model(path)
    ]
    if not completed_exports:
        raise FileNotFoundError(
            "No complete merged model export was found. Run the save-and-merge cell successfully first; "
            f"exports should be saved under {OUTPUT_DIR.resolve()}."
        )
    MERGED_MODEL_DIR = max(completed_exports, key=lambda path: path.stat().st_mtime)

print(f"Merged model source: {MERGED_MODEL_DIR.resolve()}")
llama_cpp_dir = Path.home() / "llama.cpp"
converter_path = llama_cpp_dir / "convert_hf_to_gguf.py"
if not converter_path.is_file():
    raise FileNotFoundError(
        f"Could not find {converter_path}. Clone https://github.com/ggml-org/llama.cpp "
        "to that location and install its requirements.txt before running this cell."
    )

gguf_path = OUTPUT_DIR / f"{MERGED_MODEL_DIR.name}.f16.gguf"
convert_result = run_captured_utf8(
    [
        sys.executable,
        str(converter_path),
        str(MERGED_MODEL_DIR.resolve()),
        "--outfile",
        str(gguf_path.resolve()),
        "--outtype",
        "f16",
    ],
    cwd=llama_cpp_dir,
)
if convert_result.returncode != 0:
    raise RuntimeError(
        f"Hugging Face-to-GGUF conversion failed with exit code {convert_result.returncode}.\n"
        f"stdout:\n{convert_result.stdout}\n"
        f"stderr:\n{convert_result.stderr}"
    )
if not gguf_path.is_file():
    raise FileNotFoundError(f"Conversion finished but GGUF file was not created: {gguf_path.resolve()}")
print(convert_result.stdout)
print(f"GGUF model created: {gguf_path.resolve()}")

modelfile_path = OUTPUT_DIR / f"{OLLAMA_MODEL_NAME}-Modelfile"
modelfile_path.write_text(
    f'FROM "{gguf_path.resolve().as_posix()}"\n'
    'SYSTEM """\n'
    "You are a helpful pharmaceutical assistant. Give clear, cautious educational information. "
    "Do not diagnose, prescribe, or replace advice from a qualified healthcare professional.\n"
    '"""\n'
    "PARAMETER temperature 0.3\n",
    encoding="utf-8",
)

create_result = run_captured_utf8(
    ["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(modelfile_path.resolve())]
)
if create_result.returncode != 0:
    raise RuntimeError(
        f"ollama create failed with exit code {create_result.returncode}.\n"
        f"stdout:\n{create_result.stdout}\n"
        f"stderr:\n{create_result.stderr}"
    )
print(create_result.stdout)
print(f"Created Ollama model: {OLLAMA_MODEL_NAME}")


Merged model source: C:\Users\sridh\OneDrive\Documents\LLM learning\Complete-LLM-Finetuning\LLM Fine-Tuning-16-Preference-based-training\runs\qwen2.5-1.5b-dpo\ollama-model-20261005-202227-444631


Exception in thread Thread-9 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'ch


GGUF model created: C:\Users\sridh\OneDrive\Documents\LLM learning\Complete-LLM-Finetuning\LLM Fine-Tuning-16-Preference-based-training\runs\qwen2.5-1.5b-dpo\ollama-model-20261005-202227-444631.f16.gguf


Exception in thread Thread-11 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "c:\Users\sridh\anaconda3\envs\llm_engineering\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'c


Created Ollama model: pharma-qwen-dpo


## Test the trained model locally

Ollama runs the imported model on your machine. The prompt here is a simple smoke test; medically important answers still need expert review.


In [ ]:
test_prompt = "Explain how Metformin works in simple terms."
result = subprocess.run(
    ["ollama", "run", OLLAMA_MODEL_NAME, test_prompt],
    check=True,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(result.stdout)
if result.stderr.strip():
    print(result.stderr)


## Notes

- This workflow downloads a base model from Hugging Face, trains locally using your chosen/rejected pairs, then runs the merged result with Ollama.
- Ollama `Modelfile` instructions alone only change prompt/system behavior; they do **not** train model weights.
- The sample CSV has five rows. For meaningful preference alignment, prepare many more carefully reviewed examples and keep a held-out evaluation set.
- If CUDA/4-bit package setup fails, set `USE_4BIT = False` to try standard LoRA (requires substantially more memory), or use a Linux/WSL2 CUDA environment.
- The exported model is a separate copy under `runs/qwen2.5-1.5b-dpo/ollama-model`; the base model and original DPO notebook are left unchanged.
